# phl_risk.modeling.plan 使用手册
## 本册：DataPlan 全功能详解 · ModelPlan 留待补充

**适用对象：** 希望把真实字段的职责、特征类型和未来分区方式写成稳定声明的开发者。
**范围：** 当前仓库 DataPlan 全部公开能力。所有案例仅构造声明，不读取业务数据、不执行切分、不训练模型。
**源码核对日期：** 2026-09-26。类所在模块路径也会在环境单元格中展示；相同版本号不代表源码完全相同。

DataPlan 与 ModelPlan 是两个独立构造的方案对象。你不需要先创建 ModelPlan，才能声明 RoleSpec、FeatureSpec、SplitSpec 或调用 DataPlan.validate()。
只有主动调用 `validate_against(model_plan)` 时才发生联合校验。本册只解释这个边界，不创建 ModelPlan，也不展开它的功能。

**使用方法：** 选择 Python 内核，Restart Kernel → Run All。按顺序运行；预期的错误由教程辅助函数捕获、验证并展示，不会中断整本执行。
已保存执行输出，打开 notebook 即可阅读。默认不写配置文件、不自动安装依赖、不访问网络。

## 功能树与阅读路线

- **DataPlan：数据组织方案** → [构造](#s3)、[校验边界](#s13)、[展示与导出](#s14)
  - **RoleSpec：字段扮演什么角色** → [绑定与查询](#s4)、[约束与错误](#s5)
  - **FeatureSpec：哪些字段作为特征** → [类型分组与顺序](#s6)
  - **SplitSpec：未来怎样分区** → [组合规则](#s7)
    - **PartitionSpec：最终有哪些分区** → [命名、取值、顺序](#s8)
    - **HashSplitter** → [按键成组分配声明](#s9)
    - **RandomSplitter** → [随机与分层声明](#s10)
    - **ColumnSplitter** → [已有分区字段映射](#s11)
    - **TimeSplitter** → [按时间先后比例分区声明](#s12)
    - **BaseSplitter** → [自定义声明接口](#s17)
- **不可变与复用** → [安全修改](#s15)、[完整配方](#s16)
- **排错与速查** → [错误定位](#s18)、[API 索引](#s19)、[执行验收](#s20)
- **ModelPlan** → [预留空间](#modelplan-reserved)

建议第一次按 1–20 节阅读；之后可以通过链接查参数和错误边界。表格中的“允许”仅表示声明层允许，不自动代表训练方案或执行后端接受。

<a id="s1"></a>

## 1. 环境、导入与辅助展示


推荐 Python 3.12+。在仓库根目录执行 `uv sync --group dev`，在 notebook 编辑器中选择项目 `.venv` 内核。
本册业务声明仅使用标准库和 phl_risk；表格展示使用 Jupyter 通常已提供的 IPython。

若需要命令行重跑并保存输出，可在仓库根目录执行以下命令。依赖放在临时工具环境中，不要求修改项目依赖：

```bash
uv run --no-project --python 3.12 --with nbformat --with nbclient --with nbconvert --with ipykernel \
  jupyter nbconvert --to notebook --execute --inplace examples/modeling_plan_complete_guide.ipynb
```

下面从当前工作目录及其上级定位仓库源码，适用于从根目录或 examples 目录启动。单独分发 notebook 时，应先安装对应版本的 phl-risk。

In [1]:
import inspect
import json
import sys
from dataclasses import FrozenInstanceError, dataclass, replace
from html import escape
from pathlib import Path
from typing import ClassVar, Literal

from IPython.display import HTML, display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src/phl_risk/modeling/plan/data_plan").is_dir():
        source_dir = str(candidate / "src")
        if source_dir not in sys.path:
            sys.path.insert(0, source_dir)
        break

import phl_risk  # noqa: E402 — 源码路径配置后导入
import phl_risk.modeling.plan.data_plan as data_api  # noqa: E402
from phl_risk.exceptions import (  # noqa: E402
    CompatibilityError,
    ModelingError,
    PhlRiskError,
    PlanError,
)
from phl_risk.modeling.plan.data_plan import (  # noqa: E402
    BaseSplitter,
    ColumnSplitter,
    DataPlan,
    FeatureSpec,
    HashSplitter,
    PartitionSpec,
    RandomSplitter,
    RoleSpec,
    SplitSpec,
    TimeSplitter,
)

print("Python:", sys.version.split()[0])
print("phl-risk:", phl_risk.__version__)
print("DataPlan module:", DataPlan.__module__)
print("Source:", inspect.getfile(DataPlan))

Python: 3.12.13
phl-risk: 0.5.0
DataPlan module: phl_risk.modeling.plan.data_plan.definition
Source: /Users/xuetao/git_code/phi_risk/src/phl_risk/modeling/plan/data_plan/definition.py


### 本手册的展示工具

`show_table()` 只生成小型 HTML 表格；`show_json()` 只展示 JSON；`expect_error()` 验证示例确实产生指定异常。
它们是教程辅助函数，不是 phl_risk 的 API。失败示例若没有按预期失败，辅助函数会主动抛出 AssertionError，避免“看起来通过”。

In [2]:
def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


def show_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))


expected_errors = []


def expect_error(label, exception_type, operation, contains=None):
    try:
        operation()
    except exception_type as error:
        message = str(error)
        if contains is not None:
            assert contains in message, (label, message)
        expected_errors.append((label, type(error).__name__))
        print(f"{label} → {type(error).__name__}: {message}")
    else:
        raise AssertionError(f"{label}: 没有发生预期的 {exception_type.__name__}")

<a id="s2"></a>

## 2. 先建立概念：声明不持有数据


DataPlan 的结构固定为三部分：

```text
DataPlan
├── roles: RoleSpec             字段的语义职责
├── features: FeatureSpec       数值/类别特征
└── split: SplitSpec            分配规则 + 分区定义
    ├── splitter: BaseSplitter 的具体声明
    └── partitions: PartitionSpec
```

`data_family` 固定为 `"tabular"`，不是构造参数。DataPlan 不要求传入 ModelPlan，也不保存 DataFrame。
下面的“字段字典”只是帮助理解列名，不构造真实样本数据：

In [3]:
show_table(
    ["列名", "业务含义", "在声明中的用途"],
    [
        ("user_id", "用户标识", "sample_key 的组成部分；也可单独作为 hash key"),
        ("dt", "观察日期", "sample_key 的组成部分；time 角色"),
        ("bad_30d", "未来 30 天风险标记", "target 角色"),
        ("sample_weight", "样本权重", "weight 角色"),
        ("age / income", "年龄 / 收入", "numerical 特征"),
        ("channel", "来源渠道", "categorical 特征"),
        ("dataset_flag", "上游已分好的样本集标记", "ColumnSplitter 的 source column"),
    ],
)

列名,业务含义,在声明中的用途
user_id,用户标识,sample_key 的组成部分；也可单独作为 hash key
dt,观察日期,sample_key 的组成部分；time 角色
bad_30d,未来 30 天风险标记,target 角色
sample_weight,样本权重,weight 角色
age / income,年龄 / 收入,numerical 特征
channel,来源渠道,categorical 特征
dataset_flag,上游已分好的样本集标记,ColumnSplitter 的 source column


<a id="s3"></a>

## 3. DataPlan 构造、属性和最小完整例子


签名为 `DataPlan(roles: RoleSpec, features: FeatureSpec, split: SplitSpec)`。三个参数都必填，推荐显式关键字调用。
所有子声明在构造时检查自身规则，DataPlan 构造时还会调用 validate()。成功的 validate() 返回 None，不返回处理后的数据。

In [4]:
roles = RoleSpec(
    target="bad_30d",
    sample_key=["user_id", "dt"],
    time="dt",
    weight="sample_weight",
)
features = FeatureSpec(numerical=["age", "income"], categorical=["channel"])
hash_split = SplitSpec(
    splitter=HashSplitter(key=["user_id"], random_state=2026),
    partitions=PartitionSpec(train=0.7, test=0.3),
)
data_plan = DataPlan(roles=roles, features=features, split=hash_split)

assert data_plan.validate() is None
assert data_plan.data_family == "tabular"
assert data_plan.roles is roles
assert data_plan.features.all == ("age", "income", "channel")
show_table(
    ["属性", "当前值"],
    [
        ("data_family", data_plan.data_family),
        ("roles.bindings", dict(data_plan.roles.bindings)),
        ("features.all", data_plan.features.all),
        ("split.splitter", data_plan.split.splitter),
        ("split.partitions.definitions", dict(data_plan.split.partitions.definitions)),
    ],
)

属性,当前值
data_family,tabular
roles.bindings,"{'sample_key': ('user_id', 'dt'), 'target': ('bad_30d',), 'time': ('dt',), 'weight': ('sample_weight',)}"
features.all,"('age', 'income', 'channel')"
split.splitter,"HashSplitter(key=('user_id',), random_state=2026)"
split.partitions.definitions,"{'train': 0.7, 'test': 0.3}"


### 空声明也有明确含义

`RoleSpec()` 和 `FeatureSpec()` 都合法。只有 train 的 `PartitionSpec(train=1)` 也合法。
因此可以先建立一个结构完整、尚未绑定角色和特征的方案，随后创建新版本。
这不表示方案已经满足某个学习任务，也不保证可训练。

In [5]:
minimal_plan = DataPlan(
    roles=RoleSpec(),
    features=FeatureSpec(),
    split=SplitSpec(RandomSplitter(), PartitionSpec(train=1)),
)
assert minimal_plan.validate() is None
assert minimal_plan.roles.to_dict() == {}
assert minimal_plan.features.all == ()
show_json(minimal_plan.to_dict())

{
  "data_family": "tabular",
  "roles": {},
  "features": {
    "numerical": [],
    "categorical": []
  },
  "split": {
    "splitter": {
      "type": "RandomSplitter",
      "random_state": 2026,
      "stratify": null
    },
    "partition_mode": "ratios",
    "partitions": {
      "train": 1
    }
  }
}


### 构造层级必须正确

不能把 target、time 等角色直接传给 DataPlan；不能以字典代替 Spec；也不能省略 split。
Python 的参数签名错误通常是 TypeError，声明结构错误通常是 PlanError。

In [6]:
expect_error("省略必需 split", TypeError, lambda: DataPlan(roles=roles, features=features))
expect_error(
    "把 target 放到 DataPlan 顶层",
    TypeError,
    lambda: DataPlan(target="bad_30d", roles=roles, features=features, split=hash_split),
)
for label, values in [
    ("roles 不是 RoleSpec", ({"target": "bad_30d"}, features, hash_split)),
    ("features 不是 FeatureSpec", (roles, ["age"], hash_split)),
    ("split 不是 SplitSpec", (roles, features, None)),
]:
    expect_error(label, PlanError, lambda values=values: DataPlan(*values))
expect_error(
    "data_family 不是构造参数",
    TypeError,
    lambda: DataPlan(roles, features, hash_split, data_family="sequence"),
)

省略必需 split → TypeError: DataPlan.__init__() missing 1 required positional argument: 'split'
把 target 放到 DataPlan 顶层 → TypeError: DataPlan.__init__() got an unexpected keyword argument 'target'
roles 不是 RoleSpec → PlanError: DataPlan roles must be a RoleSpec
features 不是 FeatureSpec → PlanError: DataPlan features must be a FeatureSpec
split 不是 SplitSpec → PlanError: DataPlan split must be a SplitSpec
data_family 不是构造参数 → TypeError: DataPlan.__init__() got an unexpected keyword argument 'data_family'


<a id="s4"></a>

## 4. RoleSpec：动态角色绑定与查询


`RoleSpec(**roles)` 接收动态角色名。内部 `bindings` 是只读 mapping：`semantic role → tuple[str, ...]`。
角色没有写死为 dataclass 的一组字段；新角色在声明层可以直接写入，但是否被某个模型方案接受由联合校验决定。

| 角色示例 | 含义 | 当前绑定数量 |
|---|---|---|
| target | 普通监督目标 | 1 列 |
| outcome / treatment | 结果 / 干预 | 每个角色 1 列 |
| duration / event | 持续时间 / 事件标记 | 每个角色 1 列 |
| weight | 样本权重字段 | 1 列 |
| time | 时间语义字段 | 1 列；不提供该角色即为 0 列 |
| sample_key | 联合标识一条 observation 的字段 | 至少 1 列，可以多列 |
| 自定义角色 | 由调用方定义的语义 | 当前同样限制 1 列 |

单字符串、list、tuple 都会变成 tuple；单字符串不会被拆成字符。
只有 sample_key 可绑定多列。这里声明的是字段职责，不验证目标取值、权重是否非负、时间格式或样本唯一性。

In [7]:
role_variants = {
    "单列字符串": RoleSpec(sample_key="user_id"),
    "单列列表": RoleSpec(sample_key=["user_id"]),
    "复合元组": RoleSpec(sample_key=("user_id", "dt")),
    "因果字段": RoleSpec(outcome="bad_30d", treatment="policy"),
    "生存字段": RoleSpec(duration="days", event="event_flag"),
    "自定义字段": RoleSpec(audit_group="group_id"),
}
show_table(
    ["写法", "归一化后的 bindings"],
    [(label, dict(spec.bindings)) for label, spec in role_variants.items()],
)
assert role_variants["单列字符串"] == role_variants["单列列表"]
assert role_variants["复合元组"].bindings["sample_key"] == ("user_id", "dt")

写法,归一化后的 bindings
单列字符串,"{'sample_key': ('user_id',)}"
单列列表,"{'sample_key': ('user_id',)}"
复合元组,"{'sample_key': ('user_id', 'dt')}"
因果字段,"{'outcome': ('bad_30d',), 'treatment': ('policy',)}"
生存字段,"{'duration': ('days',), 'event': ('event_flag',)}"
自定义字段,"{'audit_group': ('group_id',)}"


### 查询方式、角色顺序与缺省角色

通过 `bindings["target"]` 读取，使用 `bindings.get("weight")` 查询可选角色；不是 `roles.target`。
角色名按字典序保存，复合绑定中的列顺序保持用户输入顺序。`to_dict()` 将 tuple 转成 list 并返回独立副本。
缺省角色应当不传，不能用 None 或空列表表达“未绑定”。

In [8]:
print("target:", roles.bindings["target"])
print("没有声明的角色:", roles.bindings.get("group"))
print("角色名顺序:", list(roles.bindings))
assert list(roles.bindings) == sorted(roles.bindings)
assert RoleSpec(time="dt", target="y") == RoleSpec(target="y", time="dt")
expect_error("直接 roles.target 不存在", AttributeError, lambda: roles.target)
expect_error("下标访问缺省角色", KeyError, lambda: roles.bindings["group"])
show_json(roles.to_dict())

target: ('bad_30d',)
没有声明的角色: None
角色名顺序: ['sample_key', 'target', 'time', 'weight']
直接 roles.target 不存在 → AttributeError: 'RoleSpec' object has no attribute 'target'
下标访问缺省角色 → KeyError: 'group'
{
  "sample_key": [
    "user_id",
    "dt"
  ],
  "target": [
    "bad_30d"
  ],
  "time": [
    "dt"
  ],
  "weight": [
    "sample_weight"
  ]
}


### 同列多角色合法；sample_key 与 split key 独立

`dt` 同时作为 sample_key 的一部分与 time 是合法的。声明层也不强制其他角色绑定互不重叠。
观察粒度可以是 user_id + dt，而 hash 分配粒度仅是 user_id，以表达同一用户的多次观察应归入同一分区的意图。
这不检查真实 sample_key 唯一性，也不验证同一用户是否真的落在同一分区，因为尚未执行。

In [9]:
assert roles.bindings["sample_key"] == ("user_id", "dt")
assert roles.bindings["time"] == ("dt",)
assert data_plan.split.splitter.key == ("user_id",)
assert data_plan.split.splitter.key != roles.bindings["sample_key"]

independent_key_plan = replace(data_plan, roles=RoleSpec(target="y", sample_key="row_id"))
assert independent_key_plan.validate() is None
assert independent_key_plan.split.splitter.key == ("user_id",)
print("split key 无需等于 sample_key，也无需是它的子集。")

split key 无需等于 sample_key，也无需是它的子集。


<a id="s5"></a>

## 5. RoleSpec 的全部输入边界


角色名和列名必须是非空、非全空白的字符串。绑定只接收字符串/list/tuple，不接收 set、生成器或 None。
每个绑定内禁止重复，不会静默去重；不同角色之间可以重复使用列。
错误名称、错误类型、空绑定及错误基数在构造时立即失败。

In [10]:
for label, bindings in [
    ("空列名", {"target": ""}),
    ("全空白列名", {"target": "  "}),
    ("非字符串列", {"target": [123]}),
    ("None 不表示省略", {"weight": None}),
    ("空 sample_key", {"sample_key": []}),
    ("重复 sample_key 列", {"sample_key": ["id", "id"]}),
    ("time 不能多列", {"time": ["dt1", "dt2"]}),
    ("target 不能多列", {"target": ["y1", "y2"]}),
    ("自定义角色也不能多列", {"custom": ["a", "b"]}),
    ("set 不接受", {"sample_key": {"id"}}),
    ("生成器不接受", {"sample_key": (x for x in ["id"])}),
    ("空角色名", {"": "col"}),
]:
    expect_error(label, PlanError, lambda bindings=bindings: RoleSpec(**bindings))

空列名 → PlanError: role 'target' must be a non-empty string; got ''
全空白列名 → PlanError: role 'target' must be a non-empty string; got '  '
非字符串列 → PlanError: role 'target' must be a non-empty string; got 123
None 不表示省略 → PlanError: role 'weight' must be a column string, list or tuple
空 sample_key → PlanError: role 'sample_key' must not be empty
重复 sample_key 列 → PlanError: Duplicate columns in role 'sample_key': ('id', 'id')
time 不能多列 → PlanError: Role 'time' requires exactly one column
target 不能多列 → PlanError: Role 'target' requires exactly one column
自定义角色也不能多列 → PlanError: Role 'custom' requires exactly one column
set 不接受 → PlanError: role 'sample_key' must be a column string, list or tuple
生成器不接受 → PlanError: role 'sample_key' must be a column string, list or tuple
空角色名 → PlanError: role name must be a non-empty string; got ''


### 名称不做清洗，也不要求是 Python 标识符

实现只拒绝空/全空白字符串，并不会 strip、统一大小写或替你把空格变为下划线。
`" age "` 与 `"age"` 是两个不同列名。带空格的角色名可以通过 `**mapping` 声明，但是否适合下游应由你决定。
建议业务中采用稳定、无额外空格的列名；不要把声明通过误解为后端一定支持该名称。

In [11]:
raw_names = RoleSpec(**{"business role": " original column "})
assert raw_names.bindings["business role"] == (" original column ",)
show_json(raw_names.to_dict())

{
  "business role": [
    " original column "
  ]
}


<a id="s6"></a>

## 6. FeatureSpec：数值、类别、顺序和空集


签名为 `FeatureSpec(numerical=(), categorical=())`。两组都支持单字符串/list/tuple，默认都为空。
`numerical` 与 `categorical` 都是 tuple；`all` 始终按 numerical 在前、categorical 在后拼接。
组内禁止重复，两组禁止重叠；不同组的参数书写先后不会改变 all 的规则。

类型是调用方的声明，不是 dtype 推断。写在 categorical 不会自动编码，写在 numerical 不会自动转数值。
空特征在声明层合法，但训练后端可以拒绝。

In [12]:
feature_examples = [
    FeatureSpec(),
    FeatureSpec(numerical="age"),
    FeatureSpec(categorical=["channel", "region"]),
    FeatureSpec(categorical=["channel"], numerical=["income", "age"]),
]
show_table(
    ["numerical", "categorical", "all"],
    [(spec.numerical, spec.categorical, spec.all) for spec in feature_examples],
)
assert feature_examples[-1].all == ("income", "age", "channel")
show_json(features.to_dict())

numerical,categorical,all
(),(),()
"('age',)",(),"('age',)"
(),"('channel', 'region')","('channel', 'region')"
"('income', 'age')","('channel',)","('income', 'age', 'channel')"


{
  "numerical": [
    "age",
    "income"
  ],
  "categorical": [
    "channel"
  ]
}


### 非法特征输入

两组内的字段顺序有意义；不能用 set 代替。不要用 None 表达空组，应使用默认值或 `[]` / `()`。

In [13]:
for label, args in [
    ("数值组重复", {"numerical": ["age", "age"]}),
    ("类别组重复", {"categorical": ["channel", "channel"]}),
    ("两组重叠", {"numerical": ["age"], "categorical": ["age"]}),
    ("空列名", {"numerical": [""]}),
    ("非字符串列名", {"categorical": [None]}),
    ("None 不是空组", {"numerical": None}),
    ("set 不接受", {"numerical": {"age"}}),
]:
    expect_error(label, PlanError, lambda args=args: FeatureSpec(**args))

assert FeatureSpec(numerical=[" age ", "age"]).all == (" age ", "age")
print("特征名不会自动 strip。")

数值组重复 → PlanError: Duplicate columns in numerical: ('age', 'age')
类别组重复 → PlanError: Duplicate columns in categorical: ('channel', 'channel')
两组重叠 → PlanError: Numerical and categorical features overlap: ['age']
空列名 → PlanError: numerical must be a non-empty string; got ''
非字符串列名 → PlanError: categorical must be a non-empty string; got None
None 不是空组 → PlanError: numerical must be a column string, list or tuple
set 不接受 → PlanError: numerical must be a column string, list or tuple
特征名不会自动 strip。


<a id="s7"></a>

## 7. SplitSpec：分配规则与分区定义分开


签名为 `SplitSpec(splitter: BaseSplitter, partitions: PartitionSpec)`，两个参数必填。
SplitSpec 构造时调用 validate()；显式 validate() 成功返回 None。没有 split(df) 或 apply(df) 方法。

| 对象 | 回答的问题 |
|---|---|
| Splitter | 未来如何把 observation 分配给某个集合？ |
| PartitionSpec | 需要哪些集合，每个集合的比例或源字段匹配值是什么？ |
| SplitSpec | 这两部分放在一起是否符合声明规则？ |

SplitSpec 读取 `splitter.partition_mode` 决定如何验证 PartitionSpec，不根据数字外观猜测模式。

In [14]:
show_table(
    ["Splitter", "partition_mode", "分区定义含义"],
    [
        ("HashSplitter", HashSplitter("id").partition_mode, "比例"),
        ("RandomSplitter", RandomSplitter().partition_mode, "比例"),
        ("ColumnSplitter", ColumnSplitter("dtype").partition_mode, "源字段值"),
        ("TimeSplitter", TimeSplitter("dt").partition_mode, "按声明顺序使用的比例"),
    ],
)
assert hash_split.validate() is None
show_json(hash_split.to_dict())

Splitter,partition_mode,分区定义含义
HashSplitter,ratios,比例
RandomSplitter,ratios,比例
ColumnSplitter,values,源字段值
TimeSplitter,ratios,按声明顺序使用的比例


{
  "splitter": {
    "type": "HashSplitter",
    "key": [
      "user_id"
    ],
    "random_state": 2026
  },
  "partition_mode": "ratios",
  "partitions": {
    "train": 0.7,
    "test": 0.3
  }
}


### 对象类型和模式的错误

Splitter 必须是 BaseSplitter 的实例，不能传类对象、字符串或任意字典。
PartitionSpec 只检查通用结构；比例规则由 SplitSpec 检查。

In [15]:
expect_error("splitter 传字符串", PlanError, lambda: SplitSpec("hash", PartitionSpec(train=1)))
expect_error(
    "splitter 传类而非实例", PlanError, lambda: SplitSpec(RandomSplitter, PartitionSpec(train=1))
)
expect_error("partitions 传 dict", PlanError, lambda: SplitSpec(RandomSplitter(), {"train": 1}))

splitter 传字符串 → PlanError: SplitSpec splitter must be a BaseSplitter declaration
splitter 传类而非实例 → PlanError: SplitSpec splitter must be a BaseSplitter declaration
partitions 传 dict → PlanError: SplitSpec partitions must be a PartitionSpec


<a id="s8"></a>

## 8. PartitionSpec：名称、取值、比例和顺序


签名为 `PartitionSpec(**partitions)`，内部 `definitions` 是只读且保留插入顺序的 mapping。
至少需要名字严格等于 `train` 的分区；不强制 valid/test/oot 四件套，不限制额外名称。
`train=1` 是合法单分区。比例也可从有序字典通过 `**` 传入。

每个值允许 **Python 内置** str、int、有限 float、bool；不接受 None、list、dict、NaN、±inf。
这层允许 bool 是为了支持 ColumnSplitter 的源值标签，比例模式会进一步拒绝 bool。
名称只要求非空/非全空白，`Train` 或 `" train "` 不能替代必需的 `train`。

In [16]:
partition_examples = {
    "train/test": PartitionSpec(train=0.7, test=0.3),
    "train/valid/test": PartitionSpec(train=0.6, valid=0.2, test=0.2),
    "自定义校准集": PartitionSpec(train=0.6, calibration=0.1, holdout=0.3),
    "只有 train": PartitionSpec(train=1),
}
for label, spec in partition_examples.items():
    SplitSpec(RandomSplitter(), spec).validate()
show_table(
    ["方案", "definitions"],
    [(label, dict(spec.definitions)) for label, spec in partition_examples.items()],
)
print("访问 train:", partition_examples["train/test"].definitions["train"])
print("未提供的 valid:", partition_examples["train/test"].definitions.get("valid"))

方案,definitions
train/test,"{'train': 0.7, 'test': 0.3}"
train/valid/test,"{'train': 0.6, 'valid': 0.2, 'test': 0.2}"
自定义校准集,"{'train': 0.6, 'calibration': 0.1, 'holdout': 0.3}"
只有 train,{'train': 1}


访问 train: 0.7
未提供的 valid: None


### 比例模式的完整规则

Hash/Random/Time 的每个比例必须是内置 int/float，满足 `0 < ratio <= 1`；bool 被排除。
用 `math.fsum` 求和，再以 `math.isclose(total, 1.0, rel_tol=1e-9, abs_tol=1e-9)` 检查。
允许小量浮点误差，不会自动重标定比例，也不会自动删除比例为 0 的分区。

In [17]:
near_one = PartitionSpec(train=0.7, test=0.3000000001)
SplitSpec(HashSplitter("id"), near_one).validate()
assert near_one.definitions["test"] == 0.3000000001
print("小量浮点误差通过，但原始比例保持不变。")

for label, values in [
    ("总和小于 1", {"train": 0.6, "test": 0.3}),
    ("总和超过容差", {"train": 0.7, "test": 0.300001}),
    ("零比例", {"train": 1, "test": 0}),
    ("负比例", {"train": 1.1, "test": -0.1}),
    ("单项大于 1", {"train": 2}),
    ("bool 不是比例", {"train": True}),
    ("字符串不是比例", {"train": "0.7", "test": "0.3"}),
]:
    expect_error(
        label, PlanError, lambda values=values: SplitSpec(RandomSplitter(), PartitionSpec(**values))
    )

小量浮点误差通过，但原始比例保持不变。
总和小于 1 → PlanError: Partition ratios must sum to 1 (tolerance 1e-9)
总和超过容差 → PlanError: Partition ratios must sum to 1 (tolerance 1e-9)
零比例 → PlanError: Partition 'test' ratio must be numeric and in (0, 1]
负比例 → PlanError: Partition 'train' ratio must be numeric and in (0, 1]
单项大于 1 → PlanError: Partition 'train' ratio must be numeric and in (0, 1]
bool 不是比例 → PlanError: Partition 'train' ratio must be numeric and in (0, 1]
字符串不是比例 → PlanError: Partition 'train' ratio must be numeric and in (0, 1]


### 通用分区定义错误与重复名称

重复关键字由 Python 调用机制拒绝。若先构造 Python dict 时就覆盖了重名键，PartitionSpec 无法找回被覆盖的信息；应在调用前保证配置来源没有丢失重复项。

In [18]:
for label, values in [
    ("没有 train", {"test": 1}),
    ("Train 大小写不同", {"Train": 1}),
    ("空分区名", {"train": 0.7, "": 0.3}),
    ("空源值", {"train": ""}),
    ("None 源值", {"train": None}),
    ("复合源值", {"train": ["a", "b"]}),
    ("NaN", {"train": float("nan")}),
    ("正无穷", {"train": float("inf")}),
    ("负无穷", {"train": float("-inf")}),
]:
    expect_error(label, PlanError, lambda values=values: PartitionSpec(**values))
expect_error("重复关键字 train", TypeError, lambda: PartitionSpec(train=0.5, **{"train": 0.5}))

没有 train → PlanError: Partitions must include 'train'
Train 大小写不同 → PlanError: Partitions must include 'train'
空分区名 → PlanError: partition name must be a non-empty string; got ''
空源值 → PlanError: Partition 'train' value must be a non-empty string; got ''
None 源值 → PlanError: Partition 'train' requires a JSON scalar value
复合源值 → PlanError: Partition 'train' requires a JSON scalar value
NaN → PlanError: Partition 'train' value must be finite
正无穷 → PlanError: Partition 'train' value must be finite
负无穷 → PlanError: Partition 'train' value must be finite
重复关键字 train → TypeError: phl_risk.modeling.plan.data_plan.split.partition.PartitionSpec() got multiple values for keyword argument 'train'


### 顺序是契约的一部分

definitions 与 to_dict() 保留用户声明顺序；PartitionSpec 的相等性也考虑顺序。
即使比例相同，交换 test 与 oot 的位置也会使声明不相等。
这对 TimeSplitter 尤其重要。当前没有强制 train 必须排第一：如果你希望最早的数据作为 train，应自己把它放第一。

In [19]:
ordered = PartitionSpec(train=0.6, valid=0.2, oot=0.2)
reordered = PartitionSpec(train=0.6, oot=0.2, valid=0.2)
assert ordered != reordered
assert list(ordered.to_dict()) == ["train", "valid", "oot"]
late_train = PartitionSpec(test=0.3, train=0.7)
assert SplitSpec(TimeSplitter("dt"), late_train).validate() is None
show_table(
    ["声明", "分区顺序"],
    [
        ("常用顺序", list(ordered.definitions)),
        ("交换分区", list(reordered.definitions)),
        ("声明层允许 train 不在首位", list(late_train.definitions)),
    ],
)

声明,分区顺序
常用顺序,"['train', 'valid', 'oot']"
交换分区,"['train', 'oot', 'valid']"
声明层允许 train 不在首位,"['test', 'train']"


<a id="s9"></a>

## 9. HashSplitter：稳定按键分配的意图


签名为 `HashSplitter(key, random_state=2026)`；key 必填，支持单字符串/list/tuple，归一化为非空且不重复的列元组。
random_state 必须是非负的内置 int，不接受 bool、float、None；当前声明层没有额外的最大值限制。

Hash key 控制分配粒度：key=[user_id] 表达按用户成组，key=[user_id, dt] 表达按观察成组。
它独立于 RoleSpec.sample_key；声明层不检查字段是否存在，也不计算 hash。
字段顺序属于 key 的声明内容，不能把 set 当作 key；后端的编码协议不由此类实现。

In [20]:
hash_by_user = HashSplitter("user_id")
hash_by_observation = HashSplitter(["user_id", "dt"], random_state=42)
assert hash_by_user.key == ("user_id",)
assert hash_by_user.random_state == 2026
assert HashSplitter("id", random_state=0).random_state == 0
show_json(hash_by_observation.to_dict())

for label, args in [
    ("空 key", {"key": []}),
    ("重复 key", {"key": ["id", "id"]}),
    ("空 key 列名", {"key": " "}),
    ("负 seed", {"key": "id", "random_state": -1}),
    ("bool seed", {"key": "id", "random_state": True}),
    ("float seed", {"key": "id", "random_state": 2026.0}),
    ("None seed", {"key": "id", "random_state": None}),
]:
    expect_error(label, PlanError, lambda args=args: HashSplitter(**args))

{
  "type": "HashSplitter",
  "key": [
    "user_id",
    "dt"
  ],
  "random_state": 42
}
空 key → PlanError: HashSplitter key must not be empty
重复 key → PlanError: Duplicate columns in HashSplitter key: ('id', 'id')
空 key 列名 → PlanError: HashSplitter key must be a non-empty string; got ' '
负 seed → PlanError: random_state must be a non-negative integer
bool seed → PlanError: random_state must be a non-negative integer
float seed → PlanError: random_state must be a non-negative integer
None seed → PlanError: random_state must be a non-negative integer


<a id="s10"></a>

## 10. RandomSplitter：随机与可选分层声明


签名为 `RandomSplitter(random_state=2026, stratify=None)`。seed 的校验与 HashSplitter 一致。
stratify 只能是一个非空列名字符串，或 None；不是 True/False 开关，也不是角色名自动解析接口。
如果 target 绑定的是 bad_30d，应声明 `stratify="bad_30d"`，而不是期待 `stratify="target"` 自动找到真实列。

本阶段不生成随机数、不分层计数、不校验类别频数、不验证 train/test 均含所有类别。

In [21]:
random_plain = RandomSplitter()
random_stratified = RandomSplitter(random_state=2026, stratify="bad_30d")
random_split = SplitSpec(random_stratified, PartitionSpec(train=0.6, valid=0.2, test=0.2))
assert random_plain.stratify is None
show_json(random_split.to_dict())
for label, value in [("空 stratify", ""), ("多列 stratify", ["a", "b"]), ("布尔 stratify", True)]:
    expect_error(label, PlanError, lambda value=value: RandomSplitter(stratify=value))
expect_error("随机 seed 非整数", PlanError, lambda: RandomSplitter(random_state=1.5))

{
  "splitter": {
    "type": "RandomSplitter",
    "random_state": 2026,
    "stratify": "bad_30d"
  },
  "partition_mode": "ratios",
  "partitions": {
    "train": 0.6,
    "valid": 0.2,
    "test": 0.2
  }
}
空 stratify → PlanError: RandomSplitter stratify must be a non-empty string; got ''
多列 stratify → PlanError: RandomSplitter stratify must be a non-empty string; got ['a', 'b']
布尔 stratify → PlanError: RandomSplitter stratify must be a non-empty string; got True
随机 seed 非整数 → PlanError: random_state must be a non-negative integer


<a id="s11"></a>

## 11. ColumnSplitter：把已有字段值映射成分区


签名为 `ColumnSplitter(column)`，column 必须为非空列名。
PartitionSpec 的 **键**是方案中的分区名称，**值**是来源字段应匹配的单个标量值。
二者无需相同。例如 train="development" 表示源字段等于 development 的样本属于 train。

源值可以是字符串、内置整数、有限浮点数或 bool。数字源值不会被当成比例，也不要求相加为 1。
没有自动转字符串、范围匹配、正则匹配、多值列表匹配或兜底分区。

In [22]:
column_split = SplitSpec(
    ColumnSplitter("dataset_flag"),
    PartitionSpec(train="development", valid="validation", test="test", oot="future"),
)
show_json(column_split.to_dict())
for values in [
    {"train": 10, "test": 20},
    {"train": 0.1, "test": 0.2},
    {"train": False, "test": True},
    {"train": 1, "test": "1"},
]:
    spec = SplitSpec(ColumnSplitter("dtype"), PartitionSpec(**values))
    assert spec.validate() is None
    print("合法源值映射:", spec.partitions.to_dict())

{
  "splitter": {
    "type": "ColumnSplitter",
    "column": "dataset_flag"
  },
  "partition_mode": "values",
  "partitions": {
    "train": "development",
    "valid": "validation",
    "test": "test",
    "oot": "future"
  }
}
合法源值映射: {'train': 10, 'test': 20}
合法源值映射: {'train': 0.1, 'test': 0.2}
合法源值映射: {'train': False, 'test': True}
合法源值映射: {'train': 1, 'test': '1'}


### 重复源值与未声明源值

不同分区不能匹配 Python 认为相等的源值：1、True、1.0 彼此相等，所以不能分配到不同分区。
但数字 1 和字符串 "1" 不相等，可以分开声明。
字段不存在、真实数据含缺失值或未声明标签是执行时的问题，声明本身没有真实数据可检查。

In [23]:
for label, train_value, test_value in [
    ("相同字符串", "same", "same"),
    ("int 与 bool 等值", 1, True),
    ("int 与 float 等值", 1, 1.0),
]:
    expect_error(
        label,
        PlanError,
        lambda a=train_value, b=test_value: SplitSpec(
            ColumnSplitter("dtype"), PartitionSpec(train=a, test=b)
        ),
    )
expect_error("ColumnSplitter 空字段", PlanError, lambda: ColumnSplitter(""))
expect_error(
    "一个分区匹配多个源值不支持", PlanError, lambda: PartitionSpec(train=["a", "b"], test="c")
)

相同字符串 → PlanError: ColumnSplitter source values must be distinct across partitions
int 与 bool 等值 → PlanError: ColumnSplitter source values must be distinct across partitions
int 与 float 等值 → PlanError: ColumnSplitter source values must be distinct across partitions
ColumnSplitter 空字段 → PlanError: ColumnSplitter column must be a non-empty string; got ''
一个分区匹配多个源值不支持 → PlanError: Partition 'train' requires a JSON scalar value


<a id="s12"></a>

## 12. TimeSplitter：按时间顺序的比例声明


签名为 `TimeSplitter(time)`，time 必须显式提供真实列名；不会自动从 RoleSpec.time 读取。
当前语义是升序时间排列，然后按 PartitionSpec 声明顺序采用比例分区。
TimeSplitter 自己没有 ascending、cutoff、window、rolling 或日期区间参数。

它不排序数据、不解析日期、不定义同一时刻的样本是否可被拆开、不处理缺失时间，也不决定样本数舍入。
即使分区名为 oot，当前也只是一个名称，不会额外产生样本外时间约束。

In [24]:
time_split = SplitSpec(
    TimeSplitter(time="dt"),
    PartitionSpec(train=0.6, valid=0.2, oot=0.2),
)
assert time_split.splitter.time == "dt"
show_json(time_split.to_dict())
expect_error("TimeSplitter 必须传 time", TypeError, lambda: TimeSplitter())
expect_error("TimeSplitter 不接受空 time", PlanError, lambda: TimeSplitter(" "))
expect_error("没有 cutoff 参数", TypeError, lambda: TimeSplitter("dt", cutoff="2026-01-01"))
expect_error(
    "日期字符串不是比例",
    PlanError,
    lambda: SplitSpec(TimeSplitter("dt"), PartitionSpec(train="2025", test="2026")),
)

{
  "splitter": {
    "type": "TimeSplitter",
    "time": "dt"
  },
  "partition_mode": "ratios",
  "partitions": {
    "train": 0.6,
    "valid": 0.2,
    "oot": 0.2
  }
}
TimeSplitter 必须传 time → TypeError: TimeSplitter.__init__() missing 1 required positional argument: 'time'
TimeSplitter 不接受空 time → PlanError: TimeSplitter time must be a non-empty string; got ' '
没有 cutoff 参数 → TypeError: TimeSplitter.__init__() got an unexpected keyword argument 'cutoff'
日期字符串不是比例 → PlanError: Partition 'train' ratio must be numeric and in (0, 1]


### RoleSpec.time 与 TimeSplitter.time 当前没有一致性联检

这是现有行为边界，不是推荐写法。你可以构造一个 time 角色和时间切分列不同的方案，或者不声明 time 角色；DataPlan.validate() 仍会成功。
如果业务希望它们一致，可在自己的配置验收中加断言。不要误以为框架已经替你做了这项检查。

In [25]:
different_time_plan = DataPlan(
    roles=RoleSpec(time="observation_dt"),
    features=FeatureSpec(),
    split=time_split,
)
assert different_time_plan.validate() is None
assert different_time_plan.roles.bindings["time"] != (different_time_plan.split.splitter.time,)
print("角色时间与切分时间不同：结构合法，一致性由调用方确认。")

角色时间与切分时间不同：结构合法，一致性由调用方确认。


<a id="s13"></a>

## 13. 校验层次、独立性与执行边界


| 层次 | 入口 | 做什么 | 不做什么 |
|---|---|---|---|
| 单个声明构造 | RoleSpec / FeatureSpec / 各 Splitter / PartitionSpec | 归一化、名字/类型/局部规则校验 | 不读取数据 |
| 切分组合 | SplitSpec 构造或 validate() | splitter 类型、分区模式、比例总和或源值唯一性 | 不产生分区 |
| 数据方案 | DataPlan 构造或 validate() | 三个 Spec 类型、调用 SplitSpec.validate() | 不要求目标角色或非空特征；不全面重新构造所有子对象 |
| 可选联合校验 | DataPlan.validate_against(model_plan) | 当前模型要求的角色、数据族、类别能力 | 不训练，不检查实际样本 |
| 执行层 | 由具体后端负责 | 真实数据检查和执行 | 不在本册运行 |

RoleSpec / FeatureSpec / PartitionSpec / 各 Splitter 没有独立 validate() 方法；它们在构造时验证。
正常情况下 frozen 状态防止后续修改；不要绕过冻结机制再期待 DataPlan.validate() 修复所有状态。

### validate_against 的接口说明，暂不演示 ModelPlan

`data_plan.validate_against(model_plan) -> None` 是可选入口，参数必须是 ModelPlan 实例。
它先验证两个 Plan，然后检查并合并报告以下不匹配：

1. 缺少 required roles；
2. 提供了 required/optional 之外的角色；
3. 数据族不是模型要求的数据族；
4. 提供类别特征，但 Strategy 未声明支持 categorical。

不兼容时抛 CompatibilityError；参数类型错误时抛 PlanError。它不会把 ModelPlan 保存到 DataPlan 内部。
联合校验的具体 ModelPlan 构造与案例留到后续章节。本册不会创建模型方案、Goal 或 Strategy。

### 结构合法不代表没有数据泄漏或运行错误

下面有意声明“目标列同时作为特征、切分列同时作为特征”。当前 DataPlan 自校验允许它。
这用于说明边界，**不是建议的训练配置**。存在性、标签值、权重、泄漏、样本唯一性等必须由业务检查或执行层处理。

In [26]:
structure_only_plan = DataPlan(
    roles=RoleSpec(target="y"),
    features=FeatureSpec(numerical=["y", "id"]),
    split=SplitSpec(HashSplitter("id"), PartitionSpec(train=1)),
)
assert structure_only_plan.validate() is None
assert "y" in structure_only_plan.features.all
print("结构校验通过；尚未对真实数据或泄漏做任何检查。")
print(
    "没有 fit/predict/apply：",
    all(
        not hasattr(data_plan, method)
        for method in ("fit", "predict", "apply", "train", "transform")
    ),
)
assert isinstance(data_plan.split, SplitSpec)
assert not callable(data_plan.split)

结构校验通过；尚未对真实数据或泄漏做任何检查。
没有 fit/predict/apply： True


## Experiment 的独立边界（0.6）

以上 SplitSpec 仅声明规则，不执行切分。LightGBM Experiment 接受 prepared partitions，
不再执行 Column/Hash Splitter。用户在外部完成数据准备。
执行教程另见 [LightGBM Experiment](../docs/lightgbm_experiment.md)。

<a id="s14"></a>

## 14. to_dict、describe 与 JSON 配置


所有公开声明对象提供 to_dict()，返回仅含 dict/list/str/int/float/bool/None 的独立数据。
DataPlan 的顶层键为 data_family、roles、features、split。SplitSpec 的键为 splitter、partition_mode、partitions。
各 Splitter 的字典包含 type 标识和自身参数。

只有 DataPlan 提供本数据侧的 describe()：返回 `DataPlan` 标题加格式化 JSON 字符串，不直接打印。
子 Spec 没有 describe()，可用其 to_dict() 再自行格式化。

In [27]:
manifest = data_plan.to_dict()
assert set(manifest) == {"data_family", "roles", "features", "split"}
assert set(manifest["split"]) == {"splitter", "partition_mode", "partitions"}
assert manifest["roles"]["target"] == ["bad_30d"]
text = json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False)
assert json.loads(text) == manifest
print(data_plan.describe())

DataPlan
{
  "data_family": "tabular",
  "roles": {
    "sample_key": [
      "user_id",
      "dt"
    ],
    "target": [
      "bad_30d"
    ],
    "time": [
      "dt"
    ],
    "weight": [
      "sample_weight"
    ]
  },
  "features": {
    "numerical": [
      "age",
      "income"
    ],
    "categorical": [
      "channel"
    ]
  },
  "split": {
    "splitter": {
      "type": "HashSplitter",
      "key": [
        "user_id"
      ],
      "random_state": 2026
    },
    "partition_mode": "ratios",
    "partitions": {
      "train": 0.7,
      "test": 0.3
    }
  }
}


### 导出副本可修改，不会反向修改 Plan

角色集合按名字排序，特征与复合 key 保留列顺序，分区保留插入顺序。不要为了“稳定”对分区 mapping 重新排序，特别是 TimeSplitter。
序列化副本与声明隔离；编辑字典不会自动回写到对象，也不会自动再次校验。

In [28]:
exported = data_plan.to_dict()
exported["roles"]["sample_key"].append("extra")
exported["features"]["numerical"].append("new_feature")
exported["split"]["partitions"]["train"] = 0.5
assert data_plan.roles.bindings["sample_key"] == ("user_id", "dt")
assert data_plan.features.numerical == ("age", "income")
assert data_plan.split.partitions.definitions["train"] == 0.7
assert data_plan.to_dict() == manifest
print("修改导出数据后，原声明保持不变。")

修改导出数据后，原声明保持不变。


### 没有内置 from_dict / save / load：显式重建已知格式

JSON roundtrip 得到的是 dict，不会恢复成 DataPlan。下面仅示范**已知为 HashSplitter 的当前配置**如何显式重建，并验证 equality。
这不是通用反序列化器：其他 type 必须显式分支，不应从外部 type 字符串动态导入任意类。
也没有在本册添加 schema 版本、Artifact 或 pickle 承诺。

In [29]:
loaded = json.loads(text)
assert loaded["data_family"] == "tabular"
assert loaded["split"]["partition_mode"] == "ratios"
rule = loaded["split"]["splitter"]
assert rule["type"] == "HashSplitter"
reconstructed = DataPlan(
    roles=RoleSpec(**loaded["roles"]),
    features=FeatureSpec(**loaded["features"]),
    split=SplitSpec(
        HashSplitter(key=rule["key"], random_state=rule["random_state"]),
        PartitionSpec(**loaded["split"]["partitions"]),
    ),
)
assert reconstructed == data_plan
print("显式重建的 Hash DataPlan 与原对象相等。")

显式重建的 Hash DataPlan 与原对象相等。


如确实要保存 JSON，可在确认路径后自行执行：

```python
Path("data_plan.json").write_text(text, encoding="utf-8")
loaded = json.loads(Path("data_plan.json").read_text(encoding="utf-8"))
```

上面是可选操作，本 notebook 默认不写这个文件。

<a id="s15"></a>

## 15. 不可变对象：防御性复制与安全修改


Plan/Spec 使用 frozen dataclass，列集合归一化为 tuple，角色/分区 mapping 复制后只读。
传入列表随后被修改，不会改变已构造的对象。角色或分区的 to_dict() 是可写副本，不是内部状态的可写视图。
冻结是正常 API 使用下的约束，不是防御刻意绕过 Python 对象机制的安全边界。

In [30]:
source_key = ["user_id", "dt"]
source_features = ["age"]
source_parts = {"train": 0.7, "test": 0.3}
safe_roles = RoleSpec(sample_key=source_key)
safe_features = FeatureSpec(numerical=source_features)
safe_hash = HashSplitter(source_key)
safe_partitions = PartitionSpec(**source_parts)
source_key.append("extra")
source_features.append("extra")
source_parts["train"] = 0.5
assert safe_roles.bindings["sample_key"] == safe_hash.key == ("user_id", "dt")
assert safe_features.numerical == ("age",)
assert safe_partitions.definitions["train"] == 0.7
print("源列表和源字典变化不影响已有声明。")

源列表和源字典变化不影响已有声明。


In [31]:
expect_error(
    "不能覆盖 DataPlan.features",
    FrozenInstanceError,
    lambda: setattr(data_plan, "features", FeatureSpec()),
)
expect_error(
    "不能向 tuple 追加", AttributeError, lambda: data_plan.features.numerical.append("extra")
)


def assign_mapping(mapping, key, value):
    mapping[key] = value


expect_error(
    "不能更改 bindings", TypeError, lambda: assign_mapping(roles.bindings, "target", ("other",))
)
expect_error(
    "不能更改分区定义",
    TypeError,
    lambda: assign_mapping(hash_split.partitions.definitions, "train", 0.5),
)

不能覆盖 DataPlan.features → FrozenInstanceError: cannot assign to field 'features'
不能向 tuple 追加 → AttributeError: 'tuple' object has no attribute 'append'
不能更改 bindings → TypeError: 'mappingproxy' object does not support item assignment
不能更改分区定义 → TypeError: 'mappingproxy' object does not support item assignment


### 创建新版本，而不是原地修改

可以对 DataPlan、FeatureSpec、SplitSpec 和具体 Splitter 使用标准库 dataclasses.replace()；新的构造过程仍会执行校验。
RoleSpec 和 PartitionSpec 采用动态关键字构造，与其 dataclass 存储字段名不同，不要直接对它们用 replace()/asdict() 当通用配置接口。
通过 to_dict() 拷贝配置，修改后显式重建最清楚。整体导出统一使用 to_dict()，不保证通用 deepcopy/pickle/asdict 可用。

In [32]:
new_roles_config = roles.to_dict()
new_roles_config["target"] = ["bad_60d"]
new_roles_config.pop("weight")
new_roles = RoleSpec(**new_roles_config)
new_features = replace(features, numerical=["age", "income", "tenure"])
new_partitions = PartitionSpec(train=0.6, valid=0.2, test=0.2)
new_split = replace(
    hash_split, splitter=replace(hash_split.splitter, random_state=42), partitions=new_partitions
)
new_plan = replace(data_plan, roles=new_roles, features=new_features, split=new_split)
assert new_plan.validate() is None
assert new_plan.roles.bindings["target"] == ("bad_60d",)
assert "weight" not in new_plan.roles.bindings
assert data_plan.roles.bindings["target"] == ("bad_30d",)
assert data_plan.split.splitter.random_state == 2026
show_table(
    ["项目", "原声明", "新声明"],
    [
        ("target", data_plan.roles.bindings["target"], new_plan.roles.bindings["target"]),
        ("特征", data_plan.features.all, new_plan.features.all),
        (
            "分区",
            dict(data_plan.split.partitions.definitions),
            dict(new_plan.split.partitions.definitions),
        ),
    ],
)

项目,原声明,新声明
target,"('bad_30d',)","('bad_60d',)"
特征,"('age', 'income', 'channel')","('age', 'income', 'tenure', 'channel')"
分区,"{'train': 0.7, 'test': 0.3}","{'train': 0.6, 'valid': 0.2, 'test': 0.2}"


<a id="s16"></a>

## 16. 完整配方：按需求组合已有声明


这些是可直接修改列名后复用的 DataPlan 配方。它们相互独立，不需要 ModelPlan。
因果/生存配方仅展示领域角色如何绑定，不保证某个 Strategy 已支持相应学习问题。

In [33]:
recipes = {
    "重复观察按用户分组": DataPlan(
        RoleSpec(target="bad", sample_key=["uid", "dt"], time="dt"),
        FeatureSpec(numerical=["age", "income"], categorical=["channel"]),
        SplitSpec(HashSplitter("uid"), PartitionSpec(train=0.7, test=0.3)),
    ),
    "复用上游分区": DataPlan(
        RoleSpec(target="bad", weight="w"),
        FeatureSpec(numerical=["score"]),
        SplitSpec(ColumnSplitter("dtype"), PartitionSpec(train=0, valid=1, oot=2)),
    ),
    "随机分层意图": DataPlan(
        RoleSpec(target="label"),
        FeatureSpec(numerical=["x"]),
        SplitSpec(RandomSplitter(42, stratify="label"), PartitionSpec(train=0.8, test=0.2)),
    ),
    "时间先后意图": DataPlan(
        RoleSpec(target="bad", time="dt"),
        FeatureSpec(numerical=["x"]),
        SplitSpec(TimeSplitter("dt"), PartitionSpec(train=0.6, valid=0.2, oot=0.2)),
    ),
}
for name, recipe in recipes.items():
    assert recipe.validate() is None
show_table(
    ["配方", "规则类型", "角色", "分区"],
    [
        (
            name,
            type(recipe.split.splitter).__name__,
            list(recipe.roles.bindings),
            list(recipe.split.partitions.definitions),
        )
        for name, recipe in recipes.items()
    ],
)

配方,规则类型,角色,分区
重复观察按用户分组,HashSplitter,"['sample_key', 'target', 'time']","['train', 'test']"
复用上游分区,ColumnSplitter,"['target', 'weight']","['train', 'valid', 'oot']"
随机分层意图,RandomSplitter,['target'],"['train', 'test']"
时间先后意图,TimeSplitter,"['target', 'time']","['train', 'valid', 'oot']"


In [34]:
causal_data = DataPlan(
    RoleSpec(outcome="revenue", treatment="offer", sample_key="uid"),
    FeatureSpec(numerical=["age"]),
    hash_split,
)
survival_data = DataPlan(
    RoleSpec(duration="days_to_event", event="event_flag", sample_key="uid"),
    FeatureSpec(numerical=["age"]),
    hash_split,
)
for plan in (causal_data, survival_data):
    assert plan.validate() is None
    print(plan.roles.to_dict())
print("这些配方只说明字段组织；不验证因果或生存模型可用性。")

{'outcome': ['revenue'], 'sample_key': ['uid'], 'treatment': ['offer']}
{'duration': ['days_to_event'], 'event': ['event_flag'], 'sample_key': ['uid']}
这些配方只说明字段组织；不验证因果或生存模型可用性。


<a id="s17"></a>

## 17. BaseSplitter：扩展声明，保持不执行


BaseSplitter 是抽象基类，必须实现 `partition_mode` 和 `to_dict()`。
模式只支持 ratios 或 values；SplitSpec 会根据模式应用现有校验。
推荐 frozen dataclass，把可变输入归一化为不可变值，在构造时检查自己的字段。

下面定义一个教学用 GroupSplitter：声明将同一 group 绑定到同一分区，但故意不实现分配算法。
它仅存在于本 notebook，不加入 phl_risk。声明通过不意味着 LightGBM 或其他后端支持它。

In [35]:
@dataclass(frozen=True)
class GroupSplitter(BaseSplitter):
    """教学声明：同组观察应分到同一分区，不包含执行算法。"""

    group: str
    partition_mode: ClassVar[Literal["ratios"]] = "ratios"

    def __post_init__(self) -> None:
        if not isinstance(self.group, str) or not self.group.strip():
            raise PlanError("GroupSplitter.group must be a non-empty string")

    def to_dict(self) -> dict[str, str]:
        return {"type": "GroupSplitter", "group": self.group}


custom_split = SplitSpec(GroupSplitter("household_id"), PartitionSpec(train=0.7, test=0.3))
custom_plan = replace(data_plan, split=custom_split)
assert custom_plan.validate() is None
show_json(custom_split.to_dict())

{
  "splitter": {
    "type": "GroupSplitter",
    "group": "household_id"
  },
  "partition_mode": "ratios",
  "partitions": {
    "train": 0.7,
    "test": 0.3
  }
}


### 扩展接口同样需要校验

自定义规则不能发明 SplitSpec 不认识的 partition_mode。它可以增加自己的配置，但需要明确 to_dict 的结构，且必须仍然返回 JSON 友好值。
SplitSpec 不会通用地递归审查任意自定义 to_dict 的内容；下面主动使用 json.dumps 验证。

In [36]:
expect_error("抽象类不可直接构造", TypeError, lambda: BaseSplitter())
expect_error("自定义规则空 group", PlanError, lambda: GroupSplitter(""))


@dataclass(frozen=True)
class UnsupportedMode(BaseSplitter):
    partition_mode: ClassVar[str] = "windows"

    def to_dict(self) -> dict[str, str]:
        return {"type": "UnsupportedMode"}


expect_error(
    "未知 partition_mode", PlanError, lambda: SplitSpec(UnsupportedMode(), PartitionSpec(train=1))
)
json.dumps(custom_plan.to_dict(), allow_nan=False)
print("自定义声明通过 JSON 检查；未接入任何执行后端。")

抽象类不可直接构造 → TypeError: Can't instantiate abstract class BaseSplitter without an implementation for abstract methods 'partition_mode', 'to_dict'
自定义规则空 group → PlanError: GroupSplitter.group must be a non-empty string
未知 partition_mode → PlanError: Unsupported partition mode: 'windows'
自定义声明通过 JSON 检查；未接入任何执行后端。


<a id="s18"></a>

## 18. 错误处理与常见误解


框架异常层级为 `PhlRiskError → ModelingError → PlanError → CompatibilityError`。
PlanError 同时继承 ValueError。结构输入错误可按 PlanError 捕获，整个包错误可按 PhlRiskError 捕获。
TypeError（构造签名）、KeyError（不存在的 mapping 键）、FrozenInstanceError（冻结赋值）仍遵循 Python 语义。
不要为了让生产流程继续而吞掉所有 Exception；本册的 expect_error 仅用于教学验收。

| 现象 | 原因 | 正确做法 |
|---|---|---|
| DataPlan(target=...) 报错 | 角色不在顶层 | 包入 RoleSpec |
| roles.target 不存在 | 角色是动态 mapping | roles.bindings["target"] |
| time=[] 报错 | 空绑定不是“没有角色” | 不传 time |
| stratify=True 报错 | 需要列名或 None | stratify="label" |
| train=70/test=30 报错 | 比例不是百分数整数 | 0.7 / 0.3 |
| Column 数字标签不加起来等于 1 | values 模式不是 ratios 模式 | 无需调整 |
| 同一个源值分配给两个分区报错 | 分区匹配有歧义 | 使用不同源值 |
| TimeSplitter 不能传 cutoff | 当前只有比例声明 | 等待明确的区间声明能力 |
| 验证成功但执行拒绝 | 两层职责不同 | 核对后端能力与真实数据 |
| 修改列表/导出字典没影响 Plan | 防御性复制 | 重建子声明并 replace Plan |

In [37]:
assert issubclass(CompatibilityError, PlanError)
assert issubclass(PlanError, ModelingError)
assert issubclass(ModelingError, PhlRiskError)
assert issubclass(PlanError, ValueError)
try:
    SplitSpec(RandomSplitter(), PartitionSpec(train=0.8, test=0.8))
except PlanError as error:
    print("捕获声明错误:", error)

捕获声明错误: Partition ratios must sum to 1 (tolerance 1e-9)


<a id="s19"></a>

## 19. API 速查与源码导航


以下表格从本次运行实际导入的对象读取签名，便于与后续源码变化对照。
`bindings`、`definitions` 是查询用 mapping；它们没有 attribute-style 的列/分区字段访问。

In [38]:
public_types = [
    DataPlan,
    RoleSpec,
    FeatureSpec,
    SplitSpec,
    PartitionSpec,
    BaseSplitter,
    HashSplitter,
    RandomSplitter,
    ColumnSplitter,
    TimeSplitter,
]
show_table(
    ["类型", "构造签名"], [(cls.__name__, str(inspect.signature(cls))) for cls in public_types]
)
show_table(
    ["类型", "主要属性", "公开方法"],
    [
        (
            "DataPlan",
            "roles / features / split / data_family",
            "validate / validate_against / to_dict / describe",
        ),
        ("RoleSpec", "bindings", "to_dict"),
        ("FeatureSpec", "numerical / categorical / all", "to_dict"),
        ("SplitSpec", "splitter / partitions", "validate / to_dict"),
        ("PartitionSpec", "definitions", "to_dict"),
        ("HashSplitter", "key / random_state / partition_mode", "to_dict"),
        ("RandomSplitter", "random_state / stratify / partition_mode", "to_dict"),
        ("ColumnSplitter", "column / partition_mode", "to_dict"),
        ("TimeSplitter", "time / partition_mode", "to_dict"),
        ("BaseSplitter", "抽象 partition_mode", "抽象 to_dict"),
    ],
)

类型,构造签名
DataPlan,"(roles: phl_risk.modeling.plan.data_plan.role.RoleSpec, features: phl_risk.modeling.plan.data_plan.feature.FeatureSpec, split: phl_risk.modeling.plan.data_plan.split.split_spec.SplitSpec) -> None"
RoleSpec,"(**roles: str | list[str] | tuple[str, ...]) -> None"
FeatureSpec,"(numerical: str | list[str] | tuple[str, ...] = (), categorical: str | list[str] | tuple[str, ...] = ()) -> None"
SplitSpec,"(splitter: phl_risk.modeling.plan.data_plan.split.splitter.BaseSplitter, partitions: phl_risk.modeling.plan.data_plan.split.partition.PartitionSpec) -> None"
PartitionSpec,(**partitions: str | int | float | bool) -> None
BaseSplitter,() -> None
HashSplitter,"(key: str | list[str] | tuple[str, ...], random_state: int = 2026) -> None"
RandomSplitter,"(random_state: int = 2026, stratify: str | None = None) -> None"
ColumnSplitter,(column: str) -> None
TimeSplitter,(time: str) -> None


类型,主要属性,公开方法
DataPlan,roles / features / split / data_family,validate / validate_against / to_dict / describe
RoleSpec,bindings,to_dict
FeatureSpec,numerical / categorical / all,to_dict
SplitSpec,splitter / partitions,validate / to_dict
PartitionSpec,definitions,to_dict
HashSplitter,key / random_state / partition_mode,to_dict
RandomSplitter,random_state / stratify / partition_mode,to_dict
ColumnSplitter,column / partition_mode,to_dict
TimeSplitter,time / partition_mode,to_dict
BaseSplitter,抽象 partition_mode,抽象 to_dict


### 当前源码入口

- [数据侧公开导出](../src/phl_risk/modeling/plan/data_plan/__init__.py)
- [DataPlan 定义与联合校验](../src/phl_risk/modeling/plan/data_plan/definition.py)
- [RoleSpec](../src/phl_risk/modeling/plan/data_plan/role.py)
- [FeatureSpec](../src/phl_risk/modeling/plan/data_plan/feature.py)
- [SplitSpec](../src/phl_risk/modeling/plan/data_plan/split/split_spec.py)
- [PartitionSpec](../src/phl_risk/modeling/plan/data_plan/split/partition.py)
- [BaseSplitter 与四种规则](../src/phl_risk/modeling/plan/data_plan/split/splitter.py)
- [共享列名归一化规则](../src/phl_risk/modeling/_utils.py)
- [异常体系](../src/phl_risk/exceptions.py)

本册以这些源码为准，未将未来设计意图写成已经实现的能力。

<a id="s20"></a>

## 20. 从头执行后的自检与覆盖范围


最后检查公开类型覆盖、声明独立构造、导出可序列化、数据侧不存在处理方法，以及所有预期失败示例均被准确捕获。
联合校验的参数语义已在第 13 节说明；其成功/失败的 ModelPlan 实例用例按本册范围保留到后续补充。

In [39]:
# BaseSplitter 在 public_types 中；公开列表当前共 10 个类型。
assert {cls.__name__ for cls in public_types} == set(data_api.__all__)
for plan in [
    data_plan,
    minimal_plan,
    new_plan,
    reconstructed,
    *recipes.values(),
    causal_data,
    survival_data,
    custom_plan,
]:
    assert plan.validate() is None
    json.dumps(plan.to_dict(), allow_nan=False)
    assert isinstance(plan.describe(), str)
    assert not any(
        hasattr(plan, method) for method in ("fit", "predict", "train", "apply", "transform")
    )
assert "ModelPlan" not in globals()
assert "pandas" not in sys.modules and "lightgbm" not in sys.modules
show_table(
    ["验收项目", "结果"],
    [
        ("公开类型覆盖", f"{len(public_types)} / {len(data_api.__all__)}"),
        ("预期失败示例", f"{len(expected_errors)} 个已捕获并验证"),
        ("独立 DataPlan 构造与自校验", "通过；未创建 ModelPlan"),
        ("JSON 导出与已知 Hash 格式重建", "通过"),
        ("业务数据 / 真正切分 / 训练", "均未执行"),
    ],
)

验收项目,结果
公开类型覆盖,10 / 10
预期失败示例,74 个已捕获并验证
独立 DataPlan 构造与自校验,通过；未创建 ModelPlan
JSON 导出与已知 Hash 格式重建,通过
业务数据 / 真正切分 / 训练,均未执行


<a id="modelplan-reserved"></a>

# ModelPlan 使用手册（预留）

本区域留给后续补充。本次不编写 ModelPlan、Goal、Strategy 的说明或示例。

<!-- 后续在这里添加 ModelPlan 内容及与 DataPlan 的联合校验实例。 -->